# Factor counts

This notebook aggregates the number of charges involving each aggravating and mitigating factor, both including and excluding excluded cases. Each charge is counted at most once per factor. Free-text `Other` factors are aggregated into a single row per category in the main table (counting charges with any `Other` factor); the per-instance `Other` breakdown is exported to the `Other detail` sheet of the Excel workbook.

In [1]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
from typing import Any

import pandas as pd
from dotenv import load_dotenv

repo_root = Path.cwd().resolve()
if not (repo_root / 'featureExtraction').exists():
    repo_root = repo_root.parent

for env_path in (repo_root / 'featureExtraction' / '.env', repo_root / 'featureVerification' / '.env.local', repo_root / '.env'):
    if env_path.exists():
        load_dotenv(env_path)

from evaluate_verified_sentences import get_collection

verified_collection, _ = get_collection()
query = {'is_verified': True}
projection = {'exclude': 1, 'trials': 1}
docs = list(verified_collection.find(query, projection))

def factor_label(factor_name: str | None, other_factor: str | None) -> str | None:
    if not factor_name:
        return None
    if factor_name == 'Other':
        label = (other_factor or 'Unknown').strip() or 'Unknown'
        return f'Other: {label}'
    return factor_name

# Per (category, label) counts, plus per-category count of charges that had
# any 'Other' factor (the free-text rarely repeats, so it is aggregated).
counts_all: dict[tuple[str, str], int] = defaultdict(int)
counts_excluding_excluded: dict[tuple[str, str], int] = defaultdict(int)
other_charges_all: dict[str, int] = defaultdict(int)
other_charges_excluding_excluded: dict[str, int] = defaultdict(int)

for doc in docs:
    trials = (doc.get('trials') or {}).get('trials') or []
    is_excluded = bool(doc.get('exclude'))
    for trial in trials:
        seen_factors: set[tuple[str, str]] = set()
        had_other: set[str] = set()
        for factor_type, factors in (
            ('Aggravating', trial.get('aggravating_factors') or []),
            ('Mitigating', trial.get('mitigating_factors') or []),
        ):
            for factor in factors:
                factor_name = factor.get('factor')
                label = factor_label(factor_name, factor.get('other_factor'))
                if not label:
                    continue
                key = (factor_type, label)
                if key in seen_factors:
                    continue
                seen_factors.add(key)
                counts_all[key] += 1
                if not is_excluded:
                    counts_excluding_excluded[key] += 1
                if factor_name == 'Other':
                    had_other.add(factor_type)
        for factor_type in had_other:
            other_charges_all[factor_type] += 1
            if not is_excluded:
                other_charges_excluding_excluded[factor_type] += 1

# Standard factors: one row per distinct (category, label) that is not an 'Other:' variant
standard_keys = sorted(
    {(cat, lbl) for (cat, lbl) in counts_all if not lbl.startswith('Other:')}
    | {(cat, lbl) for (cat, lbl) in counts_excluding_excluded if not lbl.startswith('Other:')}
)
rows: list[dict[str, Any]] = []
for category, label in standard_keys:
    rows.append({
        'factor_category': category,
        'factor': label,
        'count_all_charges': counts_all.get((category, label), 0),
        'count_excluding_excluded_charges': counts_excluding_excluded.get((category, label), 0),
    })

# Aggregated 'Other' row per category (charges with any 'Other' factor)
for category in sorted(set(other_charges_all) | set(other_charges_excluding_excluded)):
    rows.append({
        'factor_category': category,
        'factor': 'Other (aggregated)',
        'count_all_charges': other_charges_all.get(category, 0),
        'count_excluding_excluded_charges': other_charges_excluding_excluded.get(category, 0),
    })

counts_df = pd.DataFrame(rows).sort_values(
    ['factor_category', 'count_all_charges', 'factor'],
    ascending=[True, False, True],
).reset_index(drop=True)

# Full 'Other' free-text breakdown for the Excel export
other_detail_keys = sorted(
    {(cat, lbl) for (cat, lbl) in counts_all if lbl.startswith('Other:')}
    | {(cat, lbl) for (cat, lbl) in counts_excluding_excluded if lbl.startswith('Other:')}
)
other_detail_rows = [
    {
        'factor_category': category,
        'factor': label,
        'count_all_charges': counts_all.get((category, label), 0),
        'count_excluding_excluded_charges': counts_excluding_excluded.get((category, label), 0),
    }
    for category, label in other_detail_keys
]
other_detail_df = pd.DataFrame(other_detail_rows).sort_values(
    ['factor_category', 'count_all_charges', 'factor'],
    ascending=[True, False, True],
).reset_index(drop=True)

output_dir = repo_root / 'notebooks'
output_dir.mkdir(exist_ok=True)
try:
    with pd.ExcelWriter(output_dir / 'factor_counts.xlsx') as writer:
        counts_df.to_excel(writer, sheet_name='Factors', index=False)
        other_detail_df.to_excel(writer, sheet_name='Other detail', index=False)
except Exception as exc:
    print(f'Excel export skipped: {exc}')

print(f'Aggregated {len(counts_df)} factors ({len(other_detail_df)} "Other" free-text variants in detail sheet) across {len(docs)} judgements')
counts_df

Aggregated 28 factors (507 "Other" free-text variants in detail sheet) across 2308 judgements


,factor_category,factor,count_all_charges,count_excluding_excluded_charges
0,Aggravating,Multiple drugs,608,549
1,Aggravating,Persistent offender,461,397
2,Aggravating,Role of the defendant,253,185
3,Aggravating,Other (aggregated),222,181
4,Aggravating,Import,181,180
5,Aggravating,On bail,100,94
6,Aggravating,Refugee/Asylum,70,58
7,Aggravating,Use of minors,28,21
8,Aggravating,Suspended sentence,20,20
9,Aggravating,Export,15,15
